# 07 — Authentication with JWT + OAuth2 Password Flow

This notebook documents how authentication is added to the Credit Card Underwriting API. It is meant to be read top-to-bottom as a learning resource — each section explains **what** is being done, **why**, and **where** the code lives in the actual project.

---

## Why do we need authentication?

Without authentication, any person or script that knows the API URL can:
- Submit credit applications and read decisions
- Read the full prediction history
- Abuse the API (even with rate limiting)

Authentication ensures only **verified, logged-in users** can access any route.

---

## The approach: JWT + OAuth2 Password Flow

**JWT (JSON Web Token)** is a compact, signed token that encodes a user's identity. The server issues a token on login; the client sends it with every subsequent request.

**OAuth2 Password Flow** is the standard protocol for exchanging a username/password for a token. FastAPI has built-in support for it.

The full flow:
```
1. User registers   →  POST /auth/register  (email + password)
2. User logs in     →  POST /auth/token     (email + password) → receives JWT
3. User calls API   →  GET /health          (Authorization: Bearer <jwt>)
4. Token expires    →  User must log in again
```

---

## Files changed in the project

| File | What changes |
|---|---|
| `api/models.py` | Add `User` database table |
| `api/schemas.py` | Add `UserCreate`, `UserRead`, `Token` Pydantic schemas |
| `api/security.py` | New file — password hashing, JWT creation/verification, `get_current_user` dependency |
| `api/auth.py` | New file — `/auth/register` and `/auth/token` routes |
| `api/main.py` | Mount auth router, protect all existing routes |
| `requirements.txt` | Add `python-jose[cryptography]` and `passlib[bcrypt]` |

## Imports

All imports needed across this notebook. In the actual project these are split across the individual files they belong to.

In [ ]:
import os
import sys
import logging
from datetime import datetime, timezone, timedelta

from fastapi import APIRouter, Depends, FastAPI, HTTPException, Request, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from jose import JWTError, jwt
from passlib.context import CryptContext
from pydantic import BaseModel, field_validator
from sqlalchemy import Column, Integer, String, Boolean
from sqlalchemy.orm import Session

sys.path.insert(0, '..')
from api.database import Base, get_db

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

---

## Step 1 — Database Model (`api/models.py`)

We need to store users in the database so we can verify their credentials on login.

**Why not store the plain password?**  
If the database is ever compromised, plain passwords would be immediately readable. Instead we store a **bcrypt hash** — a one-way transformation. On login we hash the submitted password and compare the two hashes.

**Key columns:**
- `email` — unique identifier for the user, used to look them up on login
- `password_hash` — bcrypt hash of their password, never the plain text
- `is_active` — allows disabling an account without deleting it

This class is added to the existing `api/models.py` file alongside `PredictionLog`.

In [ ]:
# api/models.py — add this class below PredictionLog

class User(Base):
    __tablename__ = "users"

    id            = Column(Integer, primary_key=True, index=True)
    email         = Column(String(255), unique=True, index=True, nullable=False)
    password_hash = Column(String(128), nullable=False)
    is_active     = Column(Boolean, default=True, nullable=False)

---

## Step 2 — Pydantic Schemas (`api/schemas.py`)

Schemas define the shape of data **coming in from the client** and **going out in responses**. They are separate from the database models because what you store and what you expose to users are often different.

We need three new schemas:

| Schema | Direction | Purpose |
|---|---|---|
| `UserCreate` | Request (inbound) | Email + password submitted at registration |
| `UserRead` | Response (outbound) | User info returned after registration — **never includes the password** |
| `Token` | Response (outbound) | The JWT returned after a successful login |

**Why does `UserRead` not include `password_hash`?**  
The response model acts as a filter — Pydantic only serializes fields that are declared in the schema. Even though the `User` ORM object has a `password_hash` attribute, it will never appear in a `UserRead` response because it is not listed here.

In [ ]:
# api/schemas.py — add these classes at the bottom of the file

class UserCreate(BaseModel):
    """Payload for POST /auth/register."""
    email: str
    password: str

    @field_validator("email")
    @classmethod
    def email_not_empty(cls, v: str) -> str:
        v = v.strip().lower()
        if not v:
            raise ValueError("Email cannot be empty.")
        return v

    @field_validator("password")
    @classmethod
    def password_min_length(cls, v: str) -> str:
        if len(v) < 8:
            raise ValueError("Password must be at least 8 characters.")
        return v


class UserRead(BaseModel):
    """Returned to the client after registration. Never exposes password_hash."""
    id: int
    email: str
    is_active: bool

    # from_attributes=True lets Pydantic read directly from a SQLAlchemy ORM object
    # instead of requiring a plain dict. Without this, returning a User ORM instance
    # from a route would raise a validation error.
    model_config = {"from_attributes": True}


class Token(BaseModel):
    """Returned to the client after a successful login."""
    access_token: str
    token_type: str = "bearer"

---

## Step 3 — Security Functions (`api/security.py`)

This is the core of the auth system. It contains:

1. **Configuration** — `SECRET_KEY`, algorithm, and token expiry loaded from environment variables
2. **Password utilities** — hash a password for storage, verify a plain password against a stored hash
3. **Token utilities** — create a signed JWT, decode and verify a JWT
4. **FastAPI dependency** — `get_current_user`, which every protected route calls to enforce authentication

### Why load `SECRET_KEY` from an environment variable?

The secret key is used to **sign** every JWT. If an attacker obtains it, they can forge tokens and impersonate any user. Hard-coding it in source code is dangerous because:
- It gets committed to git history
- Anyone with repo access can read it

Using `os.environ` means the key only lives in the server environment, never in code.

Generate a strong key with:
```bash
python -c "import secrets; print(secrets.token_hex(32))"
```
Then set it before running the server:
```bash
export SECRET_KEY=<the value above>
```

### How does `get_current_user` protect routes?

FastAPI's dependency injection system (`Depends`) calls `get_current_user` **before** the route handler runs. If the token is missing, expired, or invalid, a `401 Unauthorized` is returned immediately and the route body never executes.

In [ ]:
# api/security.py — new file

# ── Configuration ─────────────────────────────────────────────────────────────

# SECRET_KEY must be set as an environment variable — never hard-code it.
# Generate one with: python -c "import secrets; print(secrets.token_hex(32))"
SECRET_KEY: str = os.environ.get("SECRET_KEY", "")
ALGORITHM: str = "HS256"              # HMAC-SHA256 — standard for single-service JWTs
ACCESS_TOKEN_EXPIRE_MINUTES: int = 30 # Tokens expire after 30 minutes

# ── Password hashing ──────────────────────────────────────────────────────────

# CryptContext manages the hashing scheme. bcrypt is the industry standard.
# deprecated="auto" means old hashes are transparently re-hashed on next login
# if the scheme is ever upgraded.
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def get_pwd_hash(password: str) -> str:
    """Hash a plain-text password for storage in the database."""
    return pwd_context.hash(password)

def verify_pwd(plain_password: str, hashed_password: str) -> bool:
    """Return True if plain_password matches the stored hash, False otherwise."""
    return pwd_context.verify(plain_password, hashed_password)

# ── JWT utilities ─────────────────────────────────────────────────────────────

# oauth2_scheme extracts the bearer token from the Authorization header.
# tokenUrl tells FastAPI where the login endpoint is — this powers the
# "Authorize" button in the /docs Swagger UI.
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/token")

def create_access_token(email: str) -> str:
    """Create a signed JWT encoding the user's email as the subject claim.
    
    The token contains:
      sub (subject) — the user's email, used to look them up on each request
      exp (expiry)  — timestamp after which the token is rejected automatically
    """
    expire = datetime.now(timezone.utc) + timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    payload = {"sub": email, "exp": expire}
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def verify_token(token: str) -> str:
    """Decode and validate a JWT. Returns the email on success, raises 401 on failure.
    
    python-jose checks the signature and the exp claim automatically.
    A JWTError is raised for any invalid token (tampered, expired, wrong key).
    """
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        email: str | None = payload.get("sub")
        if email is None:
            raise HTTPException(status_code=401, detail="Invalid token")
        return email
    except JWTError:
        raise HTTPException(status_code=401, detail="Invalid token")

# ── FastAPI dependencies ───────────────────────────────────────────────────────

def get_current_user(
    token: str = Depends(oauth2_scheme),
    db: Session = Depends(get_db),
) -> User:
    """FastAPI dependency that enforces authentication on any route it is added to.
    
    Usage in a route:
        @app.get("/health")
        def health(_: User = Depends(get_current_user)):
            ...
    
    FastAPI calls this before the route body runs. If the token is missing or
    invalid, a 401 is returned and the route never executes.
    """
    email = verify_token(token)
    user = db.query(User).filter(User.email == email).first()
    if user is None:
        raise HTTPException(status_code=401, detail="User not found")
    return user

def get_current_active_user(current_user: User = Depends(get_current_user)) -> User:
    """Extends get_current_user to also reject disabled accounts.
    
    Use this on routes where you want to block inactive users even if their
    token is still valid (e.g. after an account is suspended).
    """
    if not current_user.is_active:
        raise HTTPException(status_code=403, detail="Account is disabled")
    return current_user

---

## Step 4 — Auth Routes (`api/auth.py`)

Two public routes that do not require a token (they are how you get one):

| Route | Purpose |
|---|---|
| `POST /auth/register` | Create a new user account. Returns the created user (no password). |
| `POST /auth/token` | Log in with email + password. Returns a JWT. |

### Why use `APIRouter` instead of `app` directly?

`APIRouter` lets you define routes in a separate file with a shared `prefix` (e.g. `/auth`) and then mount them onto the main `app` in `main.py`. This keeps auth logic isolated and `main.py` clean.

### Why does `/auth/token` use form data, not JSON?

The OAuth2 spec requires the token endpoint to accept `application/x-www-form-urlencoded` (form data), not JSON. `OAuth2PasswordRequestForm` handles this automatically. This is also what the Swagger UI "Authorize" dialog sends.

In [ ]:
# api/auth.py — new file

router = APIRouter(prefix="/auth", tags=["auth"])

@router.post("/register", response_model=UserRead, status_code=201)
def register_user(user: UserCreate, db: Session = Depends(get_db)):
    """Register a new user account.
    
    - Rejects duplicate emails with a 409 Conflict.
    - Stores a bcrypt hash of the password — the plain password is never saved.
    - Returns the created user (id, email, is_active) — never the password hash.
    """
    # Reject if email is already taken
    existing_user = db.query(User).filter(User.email == user.email).first()
    if existing_user:
        raise HTTPException(
            status_code=status.HTTP_409_CONFLICT,
            detail="An account with this email already exists."
        )

    # Hash the password before storing — never store plain text
    new_user = User(
        email=user.email,
        password_hash=get_pwd_hash(user.password),
        is_active=True,
    )
    db.add(new_user)
    db.commit()
    db.refresh(new_user)  # refresh to populate the auto-generated id
    return new_user


@router.post("/token", response_model=Token)
def login_for_access_token(
    form_data: OAuth2PasswordRequestForm = Depends(),
    db: Session = Depends(get_db),
):
    """Log in and receive a JWT bearer token.
    
    Accepts: application/x-www-form-urlencoded with fields `username` and `password`.
    Note: OAuth2PasswordRequestForm always uses the field name `username` — we treat
    it as the user's email address.
    
    The error message is intentionally generic (does not say whether the email
    or password was wrong) to prevent attackers from enumerating valid emails.
    """
    # Look up the user by email (OAuth2RequestForm calls the field `username`)
    user = db.query(User).filter(User.email == form_data.username).first()

    # Reject if user not found or password does not match
    # Generic error message prevents username enumeration
    if not user or not verify_pwd(form_data.password, user.password_hash):
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Incorrect email or password.",
            headers={"WWW-Authenticate": "Bearer"},
        )

    if not user.is_active:
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="Account is disabled."
        )

    return Token(access_token=create_access_token(user.email))

---

## Step 5 — Wiring it into `main.py`

Three changes to `api/main.py`:

1. **Mount the auth router** so `/auth/register` and `/auth/token` are available
2. **Add `Depends(get_current_active_user)`** to every existing route
3. **Add a startup guard** that prevents the server from starting if `SECRET_KEY` is not set

The parameter is named `_` (or `_current_user`) to signal it is only used for its side-effect — enforcing auth — and is not referenced inside the route body.

In [ ]:
# api/main.py — summary of changes (not the full file)

# ── New imports to add ────────────────────────────────────────────────────────
# from .auth import router as auth_router
# from .security import SECRET_KEY, get_current_active_user
# from .models import PredictionLog, User   # add User here

# ── After app.add_exception_handler(...) ──────────────────────────────────────

# Refuse to start if SECRET_KEY is missing or too short.
# This catches misconfigured environments before they accept any traffic.
# if not SECRET_KEY or len(SECRET_KEY) < 32:
#     raise RuntimeError(
#         "SECRET_KEY is not set or is too short (min 32 chars). "
#         "Run: export SECRET_KEY=$(python -c 'import secrets; print(secrets.token_hex(32))')"
#     )

# Mount the auth router — adds /auth/register and /auth/token
# app.include_router(auth_router)


# ── How to protect an existing route ─────────────────────────────────────────
# Add `_: User = Depends(get_current_active_user)` to each route signature.
# FastAPI calls this dependency before the route body — no token = 401.

# Before:
# @app.get("/health", response_model=HealthResponse, tags=["ops"])
# async def health(db: Session = Depends(get_db)):

# After:
# @app.get("/health", response_model=HealthResponse, tags=["ops"])
# async def health(
#     db: Session = Depends(get_db),
#     _: User = Depends(get_current_active_user),   # ← add this line
# ):

# Apply the same change to: /predict, /predictions, /predictions/{id}, /error-codes
# The GET / HTML route can be left unprotected (browsers can't send bearer tokens).

print("See api/main.py for the full implementation.")

---

## Step 6 — Install dependencies

Add to `requirements.txt`:
```
python-jose[cryptography]
passlib[bcrypt]
```

Then install:
```bash
pip install python-jose[cryptography] passlib[bcrypt]
```

---

## Step 7 — Testing the auth flow

Once running, navigate to `http://localhost:8000/docs`.

1. **Register**: call `POST /auth/register` with an email and password
2. **Login**: click the **Authorize** button (top right), enter your email and password
3. **Call protected routes**: the Swagger UI will automatically attach the token to every request

Or with `curl`:
```bash
# Register
curl -X POST http://localhost:8000/auth/register \
  -H "Content-Type: application/json" \
  -d '{"email": "you@example.com", "password": "mypassword"}'

# Login — note: form data, not JSON
curl -X POST http://localhost:8000/auth/token \
  -d "username=you@example.com&password=mypassword"

# Call a protected route
curl http://localhost:8000/health \
  -H "Authorization: Bearer <token from above>"
```

---

## Summary

| Concept | Where it lives | What it does |
|---|---|---|
| `User` model | `models.py` | Stores email + bcrypt password hash in SQLite |
| `UserCreate` / `UserRead` / `Token` | `schemas.py` | Validate input and shape API responses |
| `get_pwd_hash` / `verify_pwd` | `security.py` | Hash passwords for storage and verify them on login |
| `create_access_token` | `security.py` | Issue a signed JWT encoding the user's email |
| `verify_token` | `security.py` | Decode and validate a JWT, return the email |
| `get_current_active_user` | `security.py` | FastAPI dependency — rejects requests without a valid token |
| `POST /auth/register` | `auth.py` | Create account, return user info |
| `POST /auth/token` | `auth.py` | Verify credentials, return JWT |
| `Depends(get_current_active_user)` | `main.py` | Added to every protected route |

---

## Step 8 — Browser Login UI (`api/templates/login.html`)

The API routes are now protected, but the existing HTML front-end at `GET /` has no way for a user to log in from a browser. We need a dedicated login page.

### Why a separate HTML page instead of a modal or inline form?

- The main page (`index.html`) immediately redirects to `/login` if no token is found — so unauthenticated users never see the app at all
- Keeping auth on its own page makes the flow clean and easy to reason about
- No external auth library needed — plain JavaScript handles everything

### Where is the token stored?

The JWT is stored in **`localStorage`** — a browser key-value store that persists across page refreshes and browser restarts (until explicitly cleared or the user logs out).

```javascript
localStorage.setItem("token", "<jwt>")  // save on login
localStorage.getItem("token")           // read on every page load
localStorage.removeItem("token")        // clear on logout
```

### Login vs Register flow

```
Register tab:
  POST /auth/register  { email, password }  →  201 Created
  POST /auth/token     form: email+password  →  { access_token }  →  stored in localStorage
  redirect to /

Login tab:
  POST /auth/token     form: email+password  →  { access_token }  →  stored in localStorage
  redirect to /
```

Registration auto-logs the user in immediately — no need to fill in the login form again after creating an account.

### Why does the login form send form data, not JSON?

`POST /auth/token` follows the OAuth2 spec which requires `application/x-www-form-urlencoded`. The field must be named `username` (OAuth2 convention) even though we treat it as an email address.

In [ ]:
# api/templates/login.html — key JavaScript logic
# (Full file lives at api/templates/login.html)

LOGIN_JS_EXPLAINED = """
// ── On page load ──────────────────────────────────────────────────────────────

// If a token already exists, skip the login page entirely and go straight to the app
if (localStorage.getItem("token")) {
  window.location.href = "/";
}

// ── Login handler ─────────────────────────────────────────────────────────────

async function handleLogin(evt) {
  evt.preventDefault();

  // /auth/token requires form-encoded data — this is the OAuth2 spec requirement.
  // The field MUST be named `username` (OAuth2 convention), even though we use email.
  const res = await fetch("/auth/token", {
    method: "POST",
    headers: { "Content-Type": "application/x-www-form-urlencoded" },
    body: `username=${encodeURIComponent(email)}&password=${encodeURIComponent(password)}`,
  });

  if (res.ok) {
    const data = await res.json();
    localStorage.setItem("token", data.access_token); // store JWT for all future requests
    window.location.href = "/";                       // go to the app
  } else {
    // Server returns a generic error — does not reveal whether email or password was wrong
    showError("login-error", data.detail);
  }
}

// ── Register handler ──────────────────────────────────────────────────────────

async function handleRegister(evt) {
  evt.preventDefault();

  // Step 1: create the account — register accepts JSON (not form data)
  const res = await fetch("/auth/register", {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ email, password }),
  });

  if (res.ok) {
    // Step 2: auto-login immediately after registration.
    // This avoids forcing the user to fill in the login form again right after signing up.
    const loginRes = await fetch("/auth/token", {
      method: "POST",
      headers: { "Content-Type": "application/x-www-form-urlencoded" },
      body: `username=${encodeURIComponent(email)}&password=${encodeURIComponent(password)}`,
    });
    const data = await loginRes.json();
    localStorage.setItem("token", data.access_token);
    window.location.href = "/";
  }
}
"""

print("Login page JavaScript flow documented above.")

---

## Step 9 — Updating the Main App Page (`api/templates/index.html`)

Now that routes require a bearer token, every request from the browser must include the JWT stored in `localStorage`. Four changes were made to `index.html`:

### Change 1 — Redirect unauthenticated users to login

The very first thing the page does is check `localStorage` for a token. If none is found, the user is immediately sent to `/login` before anything else loads.

### Change 2 — Inject the token into every HTMX request

The history table uses HTMX to poll `GET /predictions` every 5 seconds. HTMX manages these requests internally, so we can't manually add headers. Instead we use the `htmx:configRequest` event — HTMX fires this before every request it makes, giving us a chance to inject headers:

```javascript
document.addEventListener("htmx:configRequest", evt => {
  evt.detail.headers["Authorization"] = "Bearer " + TOKEN;
});
```

This single global handler covers every HTMX request on the page automatically.

### Change 3 — Add the token to the manual fetch call

`POST /predict` is a plain `fetch()` (not HTMX), so the Authorization header is added manually:

```javascript
fetch("/predict", {
  headers: {
    "Content-Type": "application/json",
    "HX-Request": "true",
    "Authorization": "Bearer " + TOKEN,  // ← added
  },
  ...
})
```

### Change 4 — Show logged-in email and logout button

The JWT payload contains the user's email in the `sub` claim. We can read it in the browser without any library — JWTs are just base64-encoded JSON:

```javascript
function parseJwtEmail(token) {
  // A JWT is three base64 parts joined by dots: header.payload.signature
  // We only need the payload (index 1)
  return JSON.parse(atob(token.split(".")[1])).sub;
}
```

Logout removes the token from `localStorage` and redirects to `/login`.

In [ ]:
# api/templates/index.html — auth additions (added at the top of the <script> block)

INDEX_AUTH_JS = """
// ── Auth guard ────────────────────────────────────────────────────────────────

// Read the token once on page load and use it for all requests below.
// If missing, send the user to /login immediately — they never see the page.
const TOKEN = localStorage.getItem("token");
if (!TOKEN) window.location.href = "/login";

// ── Show logged-in email ──────────────────────────────────────────────────────

// Decode the JWT payload to extract the email stored in the `sub` claim.
// JWTs are: base64(header).base64(payload).signature  — no library needed.
function parseJwtEmail(token) {
  try {
    return JSON.parse(atob(token.split(".")[1])).sub;
  } catch { return ""; }
}
document.getElementById("user-email").textContent = parseJwtEmail(TOKEN);

// ── HTMX auth header injection ────────────────────────────────────────────────

// htmx:configRequest fires before every request HTMX makes.
// Adding the Authorization header here means ALL HTMX requests are authenticated
// automatically — including the history table polling every 5 seconds.
document.addEventListener("htmx:configRequest", evt => {
  evt.detail.headers["Authorization"] = "Bearer " + TOKEN;
});

// ── Logout ────────────────────────────────────────────────────────────────────

// Removing the token from localStorage is all that's needed to "log out".
// The next page load will find no token and redirect to /login.
function logout() {
  localStorage.removeItem("token");
  window.location.href = "/login";
}
"""

print("index.html auth additions documented above.")

---

## Step 10 — Adding `GET /login` to `main.py`

The login page needs its own route. Unlike all other routes, it must **not** be protected — it is the page users visit *before* they have a token.

```python
@app.get("/login", response_class=HTMLResponse, include_in_schema=False)
async def login_page(request: Request):
    return templates.TemplateResponse(request, "login.html", {})
```

- `include_in_schema=False` — hides it from the `/docs` Swagger UI (it's an HTML page, not a data endpoint)
- No `Depends(get_current_active_user)` — intentionally unprotected so unauthenticated users can reach it

---

## Updated File Summary

| File | Change |
|---|---|
| `api/templates/login.html` | **New** — Sign In / Register UI, stores JWT in localStorage |
| `api/templates/index.html` | Auth guard on load, HTMX header injection, logout button |
| `api/main.py` | Added `GET /login` route (unprotected) |

---

## Complete Authentication Flow (Browser)

```
User visits /
  → JS checks localStorage for token
  → No token → redirect to /login

User fills in login form at /login
  → POST /auth/token (form-encoded: username + password)
  → Server verifies password hash, returns JWT
  → JS stores JWT in localStorage
  → redirect to /

User is now on / with a valid token
  → All fetch() calls include:  Authorization: Bearer <jwt>
  → All HTMX requests include:  Authorization: Bearer <jwt>  (via htmx:configRequest)
  → Server validates token on every request via get_current_active_user dependency

Token expires (after 30 minutes by default)
  → Next API call returns 401 Unauthorized
  → User must log in again

User clicks Sign Out
  → localStorage.removeItem("token")
  → redirect to /login
```